In [ ]:
import pandas as pd

# Read input CSV
ie = pd.read_csv('energy_environmental_inputs.csv')

# Define louver angles
louver_angles = [-40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60]

# Create a list to store new rows
new_rows = []

# Iterate over each row in the original DataFrame
for _, row in ie.iterrows():
    for angle in louver_angles:
        new_row = row.copy()     
        new_row['LA'] = angle      
        new_rows.append(new_row)  

expanded_df = pd.DataFrame(new_rows)

expanded_df.to_csv('energy_all_inputs.csv', index=False)

print("CSV written successfully with louver angles.")

In [ ]:
# Read input CSV
ie = pd.read_csv('daylight_environmental_inputs.csv')

# Define louver angles
louver_angles = [-40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60]

# Create a list to store new rows
new_rows = []

# Iterate over each row in the original DataFrame
for _, row in ie.iterrows():
    for angle in louver_angles:
        new_row = row.copy()    
        new_row['LA'] = angle        
        new_rows.append(new_row)     

expanded_df = pd.DataFrame(new_rows)

expanded_df.to_csv('daylight_all_inputs.csv', index=False)

print("CSV written successfully with louver angles.")

In [ ]:
import pandas as pd
import numpy as np
import joblib

# Load trained models
model_energy = joblib.load('energy_model.pkl')
model_daylight = joblib.load('daylight_model.pkl')
scaler_input_energy = joblib.load('scaler_input_energy.pkl')
scaler_input_daylight = joblib.load('scaler_input_daylight.pkl')
scaler_output_daylight = joblib.load('scaler_output_daylight.pkl')


# Read CSVs and select columns
ie = pd.read_csv('energy_all_inputs.csv')
id = pd.read_csv('daylight_all_inputs.csv')

inputs_energy = ie[['HD', 'DBT', 'RH', 'DSR', 'DiSR', 'GHR', 'LA']]
inputs_daylight = id[['SAl', 'SAz', 'DNI', 'DHI', 'GHI', 'LA']]

print('Energy inputs:', inputs_energy.shape)
print('Daylight inputs:', inputs_daylight.shape)

In [ ]:
# Predicting energy for all the year
input_energy_scaled = scaler_input_energy.transform(inputs_energy.values)

y_pred_energy = model_energy.predict(input_energy_scaled)
y_pred_energy = np.clip(y_pred_energy, 0, None)

y_pred_energy_df = pd.DataFrame(y_pred_energy, columns=['energy'])

final_energy_df = pd.concat([inputs_energy, y_pred_energy_df], axis=1)
final_energy_df.to_csv('energy_predictions.csv', index=False)


# Predicting daylight for all the year
input_daylight_scaled = scaler_input_daylight.transform(inputs_daylight.values)

y_pred_scaled_daylight = model_daylight.predict(input_daylight_scaled)
y_pred_daylight = scaler_output_daylight.inverse_transform(y_pred_scaled_daylight)
y_pred_daylight = np.clip(y_pred_daylight, 0, None)

y_pred_daylight_df = pd.DataFrame(y_pred_daylight, columns=['Ev', 'rUDI'])

final_daylight_df = pd.concat([inputs_daylight, y_pred_daylight_df], axis=1)
final_daylight_df.to_csv('daylight_predictions.csv', index=False)


print("Predictions saved successfully.")